# Lab Work - 9.1

## Part 1: KNN Intuition (Q.1)

### 01 Define the KNN algorithm...

**KNN Algorithm Definition:**

Given a new point x, find the k training points closest to x by distance. For classification, assign the majority class among those k neighbors. For regression, assign their mean value.

KNN is a lazy learner because it defers all computation to prediction time.

### 02 Euclidean and Manhattan distance

In [ ]:
import numpy as np

# Points A(1,2), B(4,6)
A = np.array([1, 2])
B = np.array([4, 6])

# Euclidean
euclidean = np.sqrt(np.sum((A - B)**2))
print('Euclidean distance:', euclidean)

# Manhattan
manhattan = np.sum(np.abs(A - B))
print('Manhattan distance:', manhattan)

### 03 Manual classification example

In [ ]:
import numpy as np

train_points = np.array([[1,1], [2,2], [3,1], [5,4], [4,5]])
train_labels = ['Red', 'Red', 'Blue', 'Blue', 'Blue']
Q = np.array([3,3])

distances = np.sqrt(np.sum((train_points - Q)**2, axis=1))
print('Distances:', distances)

sorted_idx = np.argsort(distances)
print('Sorted indices:', sorted_idx)

for k in [1,3,5]:
    neighbors = [train_labels[i] for i in sorted_idx[:k]]
    from collections import Counter
    pred = Counter(neighbors).most_common(1)[0][0]
    print(f'For k={k}, prediction: {pred}')

### 04 Effect of k on decision boundary

Small k (e.g., k=1) produces a very jagged, low-bias high-variance boundary. 
Large k produces a smoother, high-bias low-variance boundary.

### 05 Feature scaling importance

In [ ]:
# Numerical example
points = np.array([[0, 1000], [1, 0], [0.5, 500]])
query = np.array([0.6, 600])

dist_no_scale = np.sqrt(np.sum((points - query)**2, axis=1))
print('Without scaling:', dist_no_scale)

# Min-max scaling
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_points = scaler.fit_transform(points)
scaled_query = scaler.transform([query])
dist_scaled = np.sqrt(np.sum((scaled_points - scaled_query)**2, axis=1))
print('With scaling:', dist_scaled)

### 06 Time complexity

Brute-force: O(n * d) per query.
KD-Tree reduces to O(d log n) on average.

## Part 2: Code It (Q.2)

### 01 Implement KNN from scratch

In [ ]:
import numpy as np
from collections import Counter

def knn_predict(X_train, y_train, x_new, k):
    # Compute distances
    distances = np.sqrt(np.sum((X_train - x_new)**2, axis=1))
    # Get k nearest indices
    k_indices = np.argsort(distances)[:k]
    # Get labels
    k_nearest_labels = [y_train[i] for i in k_indices]
    # Majority vote
    most_common = Counter(k_nearest_labels).most_common(1)
    return most_common[0][0]

# Test data from Q01
X_train = np.array([[1,1],[2,2],[3,1],[5,4],[4,5]])
y_train = [0,0,1,1,1]  # Red=0, Blue=1
x_new = np.array([3,3])

for k in [1,3,5]:
    print(f'k={k}:', knn_predict(X_train, y_train, x_new, k))

### 02 Test on Iris

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Use our KNN
preds = [knn_predict(X_train_scaled, y_train.tolist(), x, 5) for x in X_test_scaled]
print('Scratch KNN accuracy:', accuracy_score(y_test, preds))

### 03 Compare with sklearn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_sk = KNeighborsClassifier(n_neighbors=5)
knn_sk.fit(X_train_scaled, y_train)
print('sklearn accuracy:', accuracy_score(y_test, knn_sk.predict(X_test_scaled)))

## Part 3: sklearn KNN (Q.3)

### 01 Breast Cancer dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

cancer = load_breast_cancer()
X_c = cancer.data
y_c = cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42, stratify=y_c)

# Without scaling
knn_no_scale = KNeighborsClassifier(n_neighbors=5)
knn_no_scale.fit(X_train_c, y_train_c)
print('No scaling accuracy:', knn_no_scale.score(X_test_c, y_test_c))

# With scaling
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

knn_scale = KNeighborsClassifier(n_neighbors=5)
knn_scale.fit(X_train_c_scaled, y_train_c)
print('With scaling accuracy:', knn_scale.score(X_test_c_scaled, y_test_c))

### 02 k-sweep

In [ ]:
ks = [1,3,5,7,9,11,15,21,31,51]
train_accs = []
test_accs = []

for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_c_scaled, y_train_c)
    train_accs.append(knn.score(X_train_c_scaled, y_train_c))
    test_accs.append(knn.score(X_test_c_scaled, y_test_c))

import pandas as pd
results = pd.DataFrame({'k': ks, 'Train Acc': train_accs, 'Test Acc': test_accs})
print(results)

### 03 Plot accuracies

In [ ]:
import matplotlib.pyplot as plt

plt.plot(ks, train_accs, label='Train')
plt.plot(ks, test_accs, label='Test')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy vs k')
plt.show()

### 04 Classification report at optimal k

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Assume optimal k=5 or check
optimal_k = 5
knn_opt = KNeighborsClassifier(n_neighbors=optimal_k)
knn_opt.fit(X_train_c_scaled, y_train_c)
y_pred = knn_opt.predict(X_test_c_scaled)

print(classification_report(y_test_c, y_pred))

# AUC
y_prob = knn_opt.predict_proba(X_test_c_scaled)[:, 1]
print('AUC-ROC:', roc_auc_score(y_test_c, y_prob))

### 05 Compare with LR and DT

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

lr = LogisticRegression()
dt = DecisionTreeClassifier(random_state=42)

for model, name in [(knn_opt, 'KNN'), (lr, 'LR'), (dt, 'DT')]:
    if name != 'KNN':
        model.fit(X_train_c_scaled, y_train_c)
    y_pred_m = model.predict(X_test_c_scaled)
    print(f'\n{name} Accuracy:', accuracy_score(y_test_c, y_pred_m))

### 06 Distance metrics

In [ ]:
for metric in ['euclidean', 'manhattan', 'minkowski']:
    knn_m = KNeighborsClassifier(n_neighbors=5, metric=metric, p=3 if metric=='minkowski' else 2)
    knn_m.fit(X_train_c_scaled, y_train_c)
    print(f'{metric} F1-weighted:', accuracy_score(y_test_c, knn_m.predict(X_test_c_scaled)))  # approx

## Part 4: Deep Intuition (Q.4)

### 01 Bias-Variance with k=1 vs k=31

k=1: Perfect train acc (low bias, high variance) -> overfitting.
k=31: Better generalization (balanced bias-variance).

### 02 Data leakage in scaling

Scaling before split leaks test info into train. Correct: fit scaler on train only.

### 03 KNN vs Logistic Regression table

| Aspect | KNN | LR |
|--------|-----|----|
| Decision boundary | Non-linear, local | Linear |
| Training time | None | O(n d) |
| Prediction time | O(n d) | O(d) |

### 04 Reduce prediction latency

1. Use KD-Tree or Ball Tree.
2. Approximate NN (e.g., FAISS).
3. Dimensionality reduction (PCA).